# Détection d'anomalies routières — Étape 3c : Autoencodeur convolutif

**Notebook :** `03c_autoencodeur.ipynb`

Dernière approche de l'étape 3, proposée par le sujet comme **approche complémentaire**.

**Le principe.** Un **autoencodeur** apprend à **reconstruire** son entrée en la faisant passer
par un goulot d'étranglement (une représentation compressée). Il ne peut pas tout mémoriser : il
apprend les structures **fréquentes** de ses images d'entraînement.

> Si on l'entraîne à reconstruire uniquement les parties **normales** des scènes routières (route,
> bas-côtés, ciel, bâtiments), il reconstruira mal ce qu'il n'a jamais appris : les **anomalies**.
> L'**erreur de reconstruction**, pixel par pixel, devient le score d'anomalie.

C'est l'idée de départ de l'article qui a créé RoadAnomaly (Lis et al., ICCV 2019, « détection
par resynthèse »), dans une version beaucoup plus simple.

| | 03a (A, B, C) | 03b (D) | 03c (E) |
|---|---|---|---|
| Apprend | ce qu'est une anomalie | les 19 classes de Cityscapes (déjà fait par NVIDIA) | à quoi ressemble une scène **normale** |
| Utilise nos masques pour apprendre | oui : cible à prédire | non | **seulement pour exclure les anomalies** de ce qu'il apprend |
| Score d'anomalie | probabilité prédite | hésitation du modèle | erreur de reconstruction |

**GPU :** recommandé. 5 entraînements (un par tour) d'environ 3 à 5 minutes chacun sur un T4.

| § | Section |
|---|---|
| 0 | Chargement |
| 1 | Données : images réduites, découpes, masque de perte |
| 2 | Architecture de l'autoencodeur |
| 3 | Perte masquée et entraînement |
| 4 | Deux scores : erreur L1 et dissimilarité SSIM |
| 5 | Lancement : test rapide, puis les 5 tours |
| 6 | Évaluation et exemples de reconstructions |
| 7 | Sauvegarde et lecture des résultats |

## 0. Chargement

Sélectionnez **en une seule fois** `RoadAnomaly.zip` et `roadanomaly_lib.py`
(et `resultats_etape3c.zip` si vous reprenez une session interrompue).

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
RESULTS = BASE / "resultats_etape3c"
for d in ["preds", "figures", "raw"]:
    (RESULTS / d).mkdir(parents=True, exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    if trouver_dataset(BASE / "data") is None or not (BASE / "roadanomaly_lib.py").exists():
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip et roadanomaly_lib.py (+ resultats_etape3c.zip si reprise) :")
        for nom in files.upload():
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(BASE / nom) as z: z.extractall(BASE / "data")
                os.remove(BASE / nom)
            elif nom.startswith("resultats_etape3c") and nom.endswith(".zip"):
                with zipfile.ZipFile(BASE / nom) as z: z.extractall(RESULTS)
                os.remove(BASE / nom)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(BASE / nom, BASE / "roadanomaly_lib.py")
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists(), "roadanomaly_lib.py manquant (dans resultats_etape2.zip)"

import importlib, json, time, copy, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, cv2
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()
import albumentations as A
from albumentations.pytorch import ToTensorV2
sys.path.insert(0, str(BASE))
import roadanomaly_lib; importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = make_splits(DATA_ROOT)
print("Dataset :", DATA_ROOT, "| appareil :", device)
if device.type != "cuda":
    warnings.warn("Pas de GPU : seul le test rapide est raisonnable sur CPU.")
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

## 1. Données

**Résolution réduite : 480 × 270.** Un autoencodeur doit reconstruire **toute** l'image, pixel par
pixel ; c'est beaucoup plus coûteux qu'une segmentation. On travaille donc à la moitié de la
résolution de travail (480 × 270). Les cartes d'erreur sont ensuite agrandies à 960 × 540 pour
être comparées aux autres modèles avec les mêmes masques.

**Pas de normalisation ImageNet.** Il n'y a pas d'encodeur pré-entraîné : les pixels sont
simplement ramenés entre 0 et 1, et la sortie (fonction sigmoïde) est aussi entre 0 et 1.

**Augmentation limitée.** Retournement horizontal et découpes 256 × 256 uniquement : modifier les
couleurs apprendrait à l'autoencodeur à reconstruire des couleurs irréalistes, ce qui ne l'aide
pas à séparer le normal de l'anormal.

**Même découpage que tous les modèles** (`roadanomaly_lib`) : au tour *k*, entraînement sur les
3 plis d'entraînement, seuil sur la validation, test sur le pli *k*.

In [ ]:
AE_W, AE_H = 480, 270
AE_CROP = 256

def ae_train_tf():
    return A.Compose([A.Resize(AE_H, AE_W), A.HorizontalFlip(p=0.5),
                      A.RandomCrop(AE_CROP, AE_CROP), ToTensorV2()])

def ae_eval_tf():
    return A.Compose([A.Resize(AE_H, AE_W),
                      A.PadIfNeeded(min_height=None, min_width=None, pad_height_divisor=16,
                                    pad_width_divisor=16, position="top_left",
                                    border_mode=cv2.BORDER_REFLECT_101),
                      ToTensorV2()])

def ae_loaders(k, batch_size=16, samples_per_image=8):
    train, val, test = fold_sets(splits, k)
    mk = lambda names, tf, spi: RoadAnomalyDataset(DATA_ROOT, names, tf, spi)
    return (DataLoader(mk(train, ae_train_tf(), samples_per_image), batch_size=batch_size,
                       shuffle=True, num_workers=2, drop_last=True),
            DataLoader(mk(val, ae_eval_tf(), 1), batch_size=1, num_workers=2),
            DataLoader(mk(test, ae_eval_tf(), 1), batch_size=1, num_workers=2))

tl, vl, _ = ae_loaders(0)
b = next(iter(tl))
x = b["image"].float() / 255
print("lot d'entraînement :", tuple(x.shape), f"valeurs {x.min():.2f} à {x.max():.2f}",
      "| masque :", tuple(b["mask"].shape))
print("image de validation :", tuple(next(iter(vl))["image"].shape))

## 2. Architecture

Un autoencodeur convolutif symétrique :

| Partie | Couches | Taille de la carte (découpe 256 × 256) |
|---|---|---|
| Encodeur | 4 blocs « convolution de pas 2 + BatchNorm + ReLU » : 3 → 32 → 64 → 128 → 256 canaux | 256 → 128 → 64 → 32 → 16 |
| Goulot | convolution 1 × 1 : 256 → **32 canaux** | 16 × 16 × 32 |
| Décodeur | 4 blocs « agrandissement × 2 + convolution + BatchNorm + ReLU », puis convolution 3 canaux + sigmoïde | 16 → 32 → 64 → 128 → 256 |

**Le goulot est le point clé.** L'image d'entrée contient 256 × 256 × 3 ≈ 197 000 valeurs ; le
goulot n'en garde que 16 × 16 × 32 ≈ 8 200, soit **24 fois moins**. L'autoencodeur est forcé de
résumer l'image : il garde ce qui est fréquent (textures de route, de végétation, de ciel) et
perd ce qui est rare.

**Pas de connexions de saut** (contrairement à U-Net) : elles laisseraient passer les détails sans
compression, et l'autoencodeur recopierait aussi bien les anomalies que le reste.

**Agrandissement + convolution** plutôt que convolution transposée : cela évite les artefacts
en « damier » sur les reconstructions.

In [ ]:
def down(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, stride=2, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

def up(ci, co):
    return nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                         nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

class ConvAutoencoder(nn.Module):
    def __init__(self, bottleneck=32):
        super().__init__()
        self.encoder = nn.Sequential(down(3, 32), down(32, 64), down(64, 128), down(128, 256),
                                     nn.Conv2d(256, bottleneck, 1))
        self.decoder = nn.Sequential(nn.Conv2d(bottleneck, 256, 1), nn.ReLU(inplace=True),
                                     up(256, 128), up(128, 64), up(64, 32), up(32, 32),
                                     nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = ConvAutoencoder()
with torch.no_grad():
    z = ae.encoder(torch.zeros(1, 3, 256, 256))
print(f"{sum(p.numel() for p in ae.parameters()) / 1e6:.2f} M paramètres | goulot : {tuple(z.shape[1:])} "
      f"= {z.numel()} valeurs pour {3 * 256 * 256} en entrée (compression x{3 * 256 * 256 / z.numel():.0f})")
del ae

## 3. Perte masquée et entraînement

**Le problème.** Les images d'entraînement contiennent elles-mêmes des anomalies (≈ 10 % des
pixels). Si l'autoencodeur apprend à les reconstruire, elles ne produiront plus d'erreur au test.

**La solution : une perte masquée.** L'erreur de reconstruction n'est comptée **que sur les pixels
normaux** des images d'entraînement :

```
perte = Σ |x − x̂| × (1 − masque) / Σ (1 − masque)
```

Les masques ne servent donc qu'à **exclure** les anomalies de ce que l'autoencodeur apprend, pas à
lui apprendre à quoi elles ressemblent. C'est une approche **semi-supervisée** : on lui dit
seulement « ceci est normal ».

On utilise l'erreur **L1** (valeur absolue) plutôt que L2 (carré) : elle produit des
reconstructions un peu moins floues.

| Réglage | Valeur |
|---|---|
| Optimiseur | Adam, taux d'apprentissage 1e-3, décroissance cosinus |
| Époques | 60 ; meilleure perte de validation (sur les pixels normaux) conservée |
| Lots | 16 découpes 256 × 256, 8 découpes par image et par époque |

In [ ]:
class AEConfig:
    epochs = 60
    batch_size = 16
    samples_per_image = 8
    lr = 1e-3

def masked_l1(rec, x, mask):
    w = 1 - mask                                       # 1 = pixel normal
    return ((rec - x).abs().mean(1, keepdim=True) * w).sum() / w.sum().clamp_min(1)

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_ae(k, cfg, verbose=True):
    set_seed(SEED + k)
    tl, vl, test_l = ae_loaders(k, cfg.batch_size, cfg.samples_per_image)
    model = ConvAutoencoder().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    hist, best, best_state, t0 = [], 1e9, None, time.time()
    for ep in range(1, cfg.epochs + 1):
        model.train(); tot, n = 0.0, 0
        for b in tl:
            x = b["image"].to(device).float() / 255; m = b["mask"].to(device)
            loss = masked_l1(model(x), x, m)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item() * len(x); n += len(x)
        sched.step()
        model.eval(); vt, vn = 0.0, 0
        with torch.no_grad():
            for b in vl:
                x = b["image"].to(device).float() / 255; m = b["mask"].to(device)
                vt += masked_l1(model(x), x, m).item(); vn += 1
        hist.append(dict(fold=k, epoch=ep, train_l1=tot / n, val_l1_normal=vt / vn))
        if vt / vn < best:
            best = vt / vn
            best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
        if verbose and (ep % 10 == 0 or ep == 1):
            print(f"  époque {ep:3d}  L1 entraînement {tot / n:.4f}  L1 validation (pixels normaux) {vt / vn:.4f}")
    model.load_state_dict(best_state)
    pd.DataFrame(hist).to_csv(RESULTS / f"history_ae_fold{k}.csv", index=False)
    return model, vl, test_l, (time.time() - t0) / 60

## 4. Deux scores d'anomalie

À partir de l'image *x* et de sa reconstruction *x̂*, on calcule deux cartes d'erreur, lissées par
un flou gaussien (une anomalie est une **région**, pas un pixel isolé) :

| Score | Calcul | Idée |
|---|---|---|
| **E1. Erreur L1** | moyenne sur les 3 canaux de \|x − x̂\| | différence de couleur pixel par pixel |
| **E2. Dissimilarité SSIM** | (1 − SSIM local) / 2, fenêtre gaussienne 11 × 11 | compare la **structure** locale (luminance, contraste, motifs) plutôt que les couleurs exactes ; moins sensible au simple flou de la reconstruction |

**Mise à l'échelle sur 256 niveaux.** Comme pour le max-logit en 03b, chaque score est ramené
entre 0 et 1 par une transformation **fixe et croissante**, calculée sur toutes les cartes
**sans regarder les masques**. Cela ne change aucun classement de pixels.

In [ ]:
def gaussian_window(size=11, sigma=1.5):
    g = torch.exp(-(torch.arange(size) - size // 2).float() ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, size, size).contiguous()

WIN = gaussian_window().to(device)

def ssim_map(x, y, C1=0.01 ** 2, C2=0.03 ** 2):
    f = lambda t: F.conv2d(t, WIN, padding=5, groups=3)
    mx, my = f(x), f(y)
    sxx, syy, sxy = f(x * x) - mx ** 2, f(y * y) - my ** 2, f(x * y) - mx * my
    s = ((2 * mx * my + C1) * (2 * sxy + C2)) / ((mx ** 2 + my ** 2 + C1) * (sxx + syy + C2))
    return s.mean(1)                                    # moyenne sur les canaux

@torch.no_grad()
def error_maps(model, loader):
    # renvoie {nom: (erreur L1, dissimilarité SSIM)} à 960x540, plus les reconstructions
    model.eval(); out, recs = {}, {}
    for b in loader:
        x = b["image"].to(device).float() / 255
        r = model(x)
        l1 = (r - x).abs().mean(1)[0, :AE_H, :AE_W]
        ds = ((1 - ssim_map(x, r)) / 2)[0, :AE_H, :AE_W]
        maps = []
        for e in (l1, ds):
            e = cv2.GaussianBlur(e.cpu().numpy(), (0, 0), 2)
            maps.append(cv2.resize(e, (WORK_W, WORK_H), interpolation=cv2.INTER_LINEAR).astype(np.float16))
        out[b["name"][0]] = maps
        recs[b["name"][0]] = (r[0, :, :AE_H, :AE_W].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    return out, recs

## 5. Lancement

### 5.1 Test rapide (≈ 1 minute sur GPU)

In [ ]:
class Quick(AEConfig):
    epochs, samples_per_image = 2, 1

m, vl, tl, minutes = train_ae(0, Quick)
maps, _ = error_maps(m, vl)
print(f"OK en {minutes:.1f} min | {len(maps)} cartes de validation, forme {next(iter(maps.values()))[0].shape}")
del m
(RESULTS / "history_ae_fold0.csv").unlink(missing_ok=True)

### 5.2 Entraînement des 5 tours

Pour chaque tour, l'autoencodeur est entraîné puis appliqué aux images de **validation** (pour le
seuil) et de **test** du tour. Les cartes brutes sont sauvegardées dans `raw/` ; un tour déjà fait
est sauté si vous reprenez une session.

In [ ]:
FOLDS = [0, 1, 2, 3, 4]
REC_EXAMPLES = {}

for k in FOLDS:
    raw = RESULTS / "raw" / f"ae_fold{k}.npz"
    if raw.exists():
        print(f"[déjà fait] tour {k}"); continue
    print(f"=== Autoencodeur — tour {k} ===")
    model, vl, tl, minutes = train_ae(k, AEConfig)
    vmaps, _ = error_maps(model, vl)
    tmaps, recs = error_maps(model, tl)
    if k == 0:
        REC_EXAMPLES = recs
        np.savez_compressed(RESULTS / "raw" / "reconstructions_fold0.npz", **recs)
    data = {}
    for split, mp in (("val", vmaps), ("test", tmaps)):
        for n, (l1, ds) in mp.items():
            data[f"{split}|{n}|l1"], data[f"{split}|{n}|ssim"] = l1, ds
    np.savez_compressed(raw, **data)
    json.dump({"minutes": minutes}, open(RESULTS / f"time_fold{k}.json", "w"))
    print(f"-> tour {k} terminé en {minutes:.1f} min\n")
    del model; torch.cuda.empty_cache()

## 6. Évaluation

Même protocole que 03a et 03b : seuil choisi sur la validation du tour, IoU / F1 / précision /
rappel sur le test du tour. On calcule aussi AUROC, AP et FPR95 sur l'ensemble des images de test
des 5 tours (chaque image n'est testée qu'une fois, par le modèle qui ne l'a pas vue).

In [ ]:
MASKS = {n: cv2.resize(load_mask(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST).astype(bool)
         for n in splits.name}
RAW = {k: dict(np.load(RESULTS / "raw" / f"ae_fold{k}.npz")) for k in FOLDS
       if (RESULTS / "raw" / f"ae_fold{k}.npz").exists()}
KEYS = {"E_ae_l1": "l1", "E_ae_ssim": "ssim"}
LABELS = {"E_ae_l1": "E1. Autoencodeur, erreur L1", "E_ae_ssim": "E2. Autoencodeur, dissimilarité SSIM"}

# mise à l'échelle fixe (sans les masques) : percentiles 0,5 % et 99,5 % de toutes les cartes de test
SCALE = {}
for key, s in KEYS.items():
    vals = np.concatenate([v.astype(np.float32).ravel()[::97] for raw in RAW.values()
                           for kk, v in raw.items() if kk.startswith("test|") and kk.endswith("|" + s)])
    SCALE[key] = np.percentile(vals, [0.5, 99.5])
to_u8 = lambda e, lo, hi: (np.clip((e.astype(np.float32) - lo) / (hi - lo), 0, 1) * 255).round().astype(np.uint8)

def hist_of(pairs):
    pos, neg = np.zeros(256, np.int64), np.zeros(256, np.int64)
    for s, m in pairs:
        pos += np.bincount(s[m], minlength=256); neg += np.bincount(s[~m], minlength=256)
    return pos, neg

def metrics_at(pos, neg, t):
    tp, fn, fp = pos[t:].sum(), pos[:t].sum(), neg[t:].sum()
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                f1=2 * tp / max(2 * tp + fp + fn, 1), iou=tp / max(tp + fp + fn, 1))

def best_threshold(pos, neg):
    tp = pos[::-1].cumsum()[::-1]; fp = neg[::-1].cumsum()[::-1]; fn = pos.sum() - tp
    return int(np.argmax((2 * tp / np.maximum(2 * tp + fp + fn, 1))[1:]) + 1)

def curve_metrics(pos, neg):
    tp = pos[::-1].cumsum(); fp = neg[::-1].cumsum()
    tpr = np.r_[0, tp / pos.sum()]; fpr = np.r_[0, fp / neg.sum()]
    auroc = np.trapezoid(tpr, fpr) if hasattr(np, "trapezoid") else np.trapz(tpr, fpr)
    ap = np.sum(np.diff(np.r_[0, tp / pos.sum()]) * tp / np.maximum(tp + fp, 1))
    return dict(AUROC=auroc, AP=ap, FPR95=fpr[np.argmax(tpr >= 0.95)])

summ, glob_pos = [], {}
for key, s in KEYS.items():
    lo, hi = SCALE[key]
    gp, gn = np.zeros(256, np.int64), np.zeros(256, np.int64)
    for k, raw in RAW.items():
        _, val, test = fold_sets(splits, k)
        V = {n: to_u8(raw[f"val|{n}|{s}"], lo, hi) for n in val}
        T = {n: to_u8(raw[f"test|{n}|{s}"], lo, hi) for n in test}
        t = best_threshold(*hist_of((V[n], MASKS[n]) for n in val))
        pv, nv = hist_of((V[n], MASKS[n]) for n in val)
        pt, nt = hist_of((T[n], MASKS[n]) for n in test)
        gp += pt; gn += nt
        vm, tm = metrics_at(pv, nv, t), metrics_at(pt, nt, t)
        np.savez_compressed(RESULTS / "preds" / f"{key}_fold{k}.npz", **T)
        minutes = json.load(open(RESULTS / f"time_fold{k}.json"))["minutes"]
        row = dict(model=key, fold=k, best_epoch=0, epochs_run=AEConfig.epochs, threshold=t / 255,
                   val_iou=vm["iou"], val_f1=vm["f1"], test_iou=tm["iou"], test_f1=tm["f1"],
                   test_precision=tm["precision"], test_recall=tm["recall"], minutes=minutes)
        json.dump(row, open(RESULTS / f"summary_{key}_fold{k}.json", "w"), indent=1)
        summ.append(row)
    glob_pos[key] = curve_metrics(gp, gn)

summ = pd.DataFrame(summ); summ["score"] = summ.model.map(LABELS)
glob = pd.DataFrame(glob_pos).T; glob.index = [LABELS[k] for k in glob.index]
print("Métriques indépendantes du seuil (images de test des 5 tours)")
display(glob.style.format("{:.3f}"))

par_tour = summ.pivot_table(index="score", columns="fold", values="test_iou").round(3)
par_tour.columns = [f"tour {c}" for c in par_tour.columns]
display(par_tour)
agg = summ.groupby("score")[["test_iou", "test_f1", "test_precision", "test_recall", "threshold"]].agg(["mean", "std"])
pm = lambda m, s: f"{m:.3f} ± {s:.3f}" if s == s else f"{m:.3f} ± —"
display(pd.DataFrame(index=agg.index, data={
    "IoU test": [pm(m, s) for m, s in zip(agg.test_iou["mean"], agg.test_iou["std"])],
    "F1 (= Dice) test": [pm(m, s) for m, s in zip(agg.test_f1["mean"], agg.test_f1["std"])],
    "précision": agg.test_precision["mean"].map("{:.3f}".format),
    "rappel": agg.test_recall["mean"].map("{:.3f}".format),
    "seuil moyen": agg.threshold["mean"].map("{:.2f}".format)}))
summ.to_csv(RESULTS / "resume_autoencodeur.csv", index=False)
glob.to_csv(RESULTS / "metriques_globales_autoencodeur.csv")

### 6.1 Courbe d'apprentissage (tour 0)

La perte de validation est mesurée **sur les pixels normaux** des images de validation : elle
indique si l'autoencodeur apprend à reconstruire des scènes qu'il n'a jamais vues.

In [ ]:
h = pd.read_csv(RESULTS / "history_ae_fold0.csv")
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(h.epoch, h.train_l1, color="#8a8984", lw=2, label="entraînement")
ax.plot(h.epoch, h.val_l1_normal, color="#2a78d6", lw=2, label="validation (pixels normaux)")
ax.set_xlabel("Époque"); ax.set_ylabel("Erreur L1 moyenne"); ax.legend(frameon=False)
ax.set_title("Erreur de reconstruction de l'autoencodeur (tour 0)", fontweight="bold")
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "01_courbe_apprentissage_ae.png"); plt.show()

### 6.2 Exemples de reconstructions (images de test du tour 0)

Colonnes : image, **reconstruction**, carte d'erreur L1, carte SSIM (blanc = erreur élevée),
contour cyan = vérité terrain.

In [ ]:
recs = dict(np.load(RESULTS / "raw" / "reconstructions_fold0.npz"))
test0 = fold_sets(splits, 0)[2]
choix = test0[:: max(1, len(test0) // 4)][:4]
lo1, hi1 = SCALE["E_ae_l1"]; los, his = SCALE["E_ae_ssim"]
cols = ["Image", "Reconstruction", "Erreur L1", "Dissimilarité SSIM"]
fig, axes = plt.subplots(len(choix), 4, figsize=(13.5, 2.1 * len(choix)))
for r, n in enumerate(choix):
    img = cv2.resize(load_image(DATA_ROOT, n), (AE_W, AE_H), interpolation=cv2.INTER_AREA)
    cont, _ = cv2.findContours(cv2.resize(MASKS[n].astype(np.uint8), (AE_W, AE_H), interpolation=cv2.INTER_NEAREST),
                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    e1 = cv2.resize(to_u8(RAW[0][f"test|{n}|l1"], lo1, hi1), (AE_W, AE_H))
    e2 = cv2.resize(to_u8(RAW[0][f"test|{n}|ssim"], los, his), (AE_W, AE_H))
    panels = [img, recs[n], cv2.cvtColor(e1, cv2.COLOR_GRAY2RGB), cv2.cvtColor(e2, cv2.COLOR_GRAY2RGB)]
    for c, p in enumerate(panels):
        axes[r, c].imshow(cv2.drawContours(p.copy(), cont, -1, (0, 200, 255), 1) if c >= 2 else p)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
        if r == 0: axes[r, c].set_title(cols[c], fontsize=9)
    axes[r, 0].set_ylabel(n[:20], fontsize=8)
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "02_exemples_reconstructions.png"); plt.show()

## 7. Sauvegarde et lecture des résultats

Les fichiers `preds/E_ae_*_fold<k>.npz` et `summary_*.json` ont le même format que 03a et 03b,
pour le benchmarking de l'étape 4.

**Comment interpréter vos chiffres :**

1. **Une reconstruction floue.** Avec un goulot 24 fois plus petit que l'entrée, la
   reconstruction perd les détails fins **partout** : contours, textures, feuillage, marquages au
   sol. L'erreur est donc élevée sur **tout ce qui est détaillé**, normal ou non. C'est la
   principale cause de fausses alarmes.
2. **Ce que l'autoencodeur détecte bien :** les objets dont la **couleur ou la luminosité**
   diffère nettement de leur entourage (un animal sombre sur une route claire).
3. **Ce qu'il manque :** les objets qui ressemblent aux textures normales (un rocher gris sur une
   route grise). Cela rejoint l'étape 1 (section 9) : la couleur seule sépare mal les anomalies
   (AUROC ≤ 0,63). Un autoencodeur de pixels apprend surtout des couleurs et des textures, pas ce
   que sont les objets.
4. **L1 ou SSIM ?** SSIM, qui compare les structures, est en général moins sensible au flou
   général de la reconstruction ; vérifiez-le sur vos AUROC et AP.
5. **Ce qu'on attend :** une performance proche ou inférieure à celle de l'approche Cityscapes
   (AP 0,42). Ce n'est pas un échec du code : c'est la limite connue de la reconstruction de pixels,
   qui a poussé les auteurs de RoadAnomaly vers la **resynthèse** à partir d'une segmentation
   (piste d'amélioration à citer dans le rapport).

In [ ]:
shutil.make_archive(str(BASE / "resultats_etape3c"), "zip", RESULTS)
if IN_COLAB:
    from google.colab import files
    files.download(str(BASE / "resultats_etape3c.zip"))
print(sorted(str(p.relative_to(RESULTS)) for p in RESULTS.rglob("*") if p.is_file())[:10], "…")